<a href="https://colab.research.google.com/github/rastri-dey/Ground-up-implementations-ML-algorithms-/blob/main/notebooks/RNN_scratch_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Description

Building a language model using RNN from scratch within Pytorch framework

**ML Algorithm**: RNN from scratch <br>
**Dataset**: Book by H G Wells "The Time Machine" <br>
**Framework**: PyTorch

## Terminologies

**tokens**: Each time step corresponds to 1 token. In a character level tokenization, each character is a token. <br>
**corpus**: corpus is a single list of token indices from the entire book, represented as numerical indices based on the vocabulary <br>
**vocab**: vocab is the vocabulary of The Time Machine corpus. In character level language modelling, it is a set of all characters in the book. (There is maximum of 256 ASCII characters). So this is bounded by 256 maximum.<br>

## Import Libraries

In [ ]:
import random
import torch
import torch.nn as nn

# Data Batch Processing

## Random Sampling of sequences within mini batches
**corpus**: Entire list of characters <br>
num_subseqs: partitioning the entire list into small subsequences of num_steps length <br>
**num_steps**: length of one subsequence <br>
initial_indices: first index of each subsequence of the entire list of characters in the book. Like if each subsequence length is 5, then this list is : `[0, 5, 10, 15, ....]` <br>
**num_batches**: how many batches of data we need. Like we may want to send the whole corpus in two batches. <br>
**Random Sampling**: Shuffle the initial_indices list randomly like `[10,0,5,15]`. So, first batch will have `[10,0]`, 2nd batch will have `[5,15]`. This ensures not only two adjacent subsequences of two different mini-batches are not really adajacent in corpus, but also 2 adjacent subsequence within same mini-batch might not be adajacent within the corpus. <br>

**`yield`** keyword within a function makes the function a generator/iterator. So when the function is being called using a for loop, at each loop it gives the yield values, pauses its execution, saves the local states until the next iteration reaches to yield

```
def simple_generator():
    yield 1
    yield 2
    yield 3

# Using the standalone generator
for value in simple_generator():
    print(value)
```

SeqDataLoader class itself is iterable because its `__iter__` method returns an iterator/generator. The seq_data_iter_random is a generator function because it uses yield, which *generates the batches of data (X,Y) in every iteration of the for loop*.

In [1]:
def seq_random_sampl(corpus, num_steps, batch_size):
  '''
  Inputs: The textbook data
  Outputs: Batch data: Input data X(sequence of characters) and corresponding labels Y(expected next sequence of characters, given last input)
  Process: Random sampling of sequences of data
  '''
  corpus = corpus[random.randint(0,num_steps-1):] # Based on the Book details, we need the corpus to start from different random starting points
  num_seqs = (len(corpus)-1)//num_steps
  initial_indices = list(range(0, num_seqs*num_steps, num_steps))
  random.shuffle(initial_indices)

  num_batches = num_seqs//batch_size

  def data(pos):
    return corpus[pos:pos+num_steps]

  for i in range(num_batches):
    rand_batch_indices = initial_indices[i:i+batch_size]
    X = [data(pos) for pos in rand_batch_indices]
    Y = [data(pos+1) for pos in rand_batch_indices]
    yield torch.tensor(X), torch.tensor(Y)

## Sequential Sampling of sequences within mini batches

In [ ]:
def seq_sequential_sampl(corpus, num_steps, batch_size):
  '''
  Inputs: The textbook data
  Outputs: Batch data: Input data X(sequence of characters) and corresponding labels Y(expected next sequence of characters, given last input)
  Process: Sequential sampling of sequences of data
  '''
  offset = random.randint(0, num_steps-1)

  num_tokens = ((len(corpus)-offset-1)//batch_size)*batch_size  # Intention is to make the num_tokens a multiple of batch size, so that matrix is even, -1 is done to consider for the final label char of final input char

  Xs = torch.tensor(corpus[offset:num_tokens])     # A list
  Ys = torch.tensor(corpus[offset+1:num_tokens+1]) # A list
  Xs = Xs.reshape(batch_size, -1)                  # Matrix would be even, because of multiple of num_tokens calculation
  Ys = Ys.reshape(batch_size, -1)

  num_batches = Xs.shape[1]//num_steps

  for i in range(num_batches):
    X = Xs[:, i : i+num_steps]
    Y = Ys[:, i : i+num_steps] # No need of doing pos+1, since its already taken in tensor list Ys
    yield X, Y